# Opdracht 4 – ETL Implementatie Data Warehouse
## Robbert & Mees

In deze opdracht implementeren wij de ETL-processen van het Data Warehouse. 
We laden data vanuit het Source Data Model (SDM) naar het DWH en passen 
Slowly Changing Dimensions Type 1 en Type 2 toe.

## Inlaadstrategie

In deze opdracht maken wij gebruik van de **Incremental Data Loading** strategie.

Reden:
- we willen alleen nieuwe en gewijzigde data verwerken
- nodig voor SCD Type 2 (historie behouden)


Setup & Connecties (NOG INVULLEN)

In [1]:
import pandas as pd
import sqlite3
from loguru import logger
from datetime import datetime # nodig voor dimensies (begin/eindtijd)

sdm_conn = sqlite3.connect('../Source Data Model/BikeToDrive_V3_OLTP.db', timeout=30)
dwh_conn = sqlite3.connect('../Data Warehouse/BikeToDrive_DWH_V2.db', timeout=30)
sdm_cursor = sdm_conn.cursor()
dwh_cursor = dwh_conn.cursor()

logger.info("SDM en DWH connecties succesvol.")

2026-03-30 16:50:48.063 | INFO     | __main__:<module>:11 - SDM en DWH connecties succesvol.


Dim_Klant (SCD TYPE 2) ROBBERT

In [ ]:
import pandas as pd
from datetime import datetime

# Wat willen we bereiken?
# We willen Dim_Klant in het DWH vullen met SCD Type 2 logica:
#
# - nieuwe klant in SDM, nog niet in DWH -> INSERT
# - bestaande klant, maar gewijzigd -> oude rij afsluiten + nieuwe rij INSERT
# - bestaande klant, niet gewijzigd -> niets doen
#
# EXTRA:
# We halen klantdata uit 2 brontabellen:
# - Accessoire_Verkoop_Klant
# - Fiets_Verkoop_Klant
#
# Daarbij kan het gebeuren dat hetzelfde klantnr in beide bronnen voorkomt.
# Dan controleren we:
# - zijn de gegevens exact gelijk? -> dan nemen we de klant 1 keer mee
# - zijn de gegevens verschillend? -> dan is dit een conflict
#   en nemen we deze klant NIET mee in de ETL
#
# business key = klantnr
# surrogate key = klant_sk

logger.info("Start ETL Dim_Klant (SCD Type 2)")

# 1. Brondata ophalen
# We lezen de klantdata uit beide brontabellen.
# Beide tabellen hebben dezelfde kolommenstructuur.

query_accessoire_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Accessoire_Verkoop_Klant
"""

query_fiets_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Fiets_Verkoop_Klant
"""

df_accessoire_klant = pd.read_sql_query(query_accessoire_klant, sdm_conn)
df_fiets_klant = pd.read_sql_query(query_fiets_klant, sdm_conn)

logger.info(f"Aantal klant-rijen uit accessoirebron: {len(df_accessoire_klant)}")
logger.info(f"Aantal klant-rijen uit fietsbron: {len(df_fiets_klant)}")

# 2. Verticale samenvoeging tot één bronset
# We zetten alle klanten uit beide bronnen onder elkaar.
# Daarna bepalen we per klantnr wat we doen:
#
# - komt klantnr maar 1 keer voor -> gewoon meenemen
# - komt klantnr meerdere keren voor:
#     - zijn alle waarden gelijk? -> 1 keer meenemen
#     - verschillen de waarden? -> conflict loggen en overslaan

df_all = pd.concat(
    [df_accessoire_klant, df_fiets_klant],
    ignore_index=True
)

df_klant_source = []
conflicts = []

for klantnr, group in df_all.groupby("klantnr"):
    if len(group) == 1:
        # klant komt maar 1 keer voor in de brondata
        df_klant_source.append(group.iloc[0])
    else:
        # klant komt meerdere keren voor
        # controleer of de gegevens gelijk zijn
        unieke_rijen = group.drop_duplicates(subset=[
            "naam",
            "adres",
            "woonplaats",
            "geslacht",
            "geboortedatum"
        ])

        if len(unieke_rijen) == 1:
            # gegevens zijn gelijk -> 1 keer meenemen
            df_klant_source.append(unieke_rijen.iloc[0])
        else:
            # gegevens verschillen -> conflict
            conflicts.append(klantnr)
            logger.warning(f"Conflict klantnr={klantnr}")

# Maak van de lijst weer een dataframe
df_klant_source = pd.DataFrame(df_klant_source)

logger.info(f"Aantal conflicten in brondata: {len(conflicts)}")
logger.info(f"Aantal unieke klanten in source: {len(df_klant_source)}")

# 3. Actuele records uit het DWH ophalen
# Voor SCD Type 2 vergelijken we alleen met de huidige actieve versie.
# Daarom halen we alleen records op waarvan eindtijd IS NULL.

query_dwh_klant = """
SELECT
    klant_sk,
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum,
    begintijd,
    eindtijd
FROM Dim_Klant
WHERE eindtijd IS NULL
"""

df_klant_dwh = pd.read_sql_query(query_dwh_klant, dwh_conn)

logger.info(f"Aantal actuele klanten in DWH: {len(df_klant_dwh)}")

# 4. Helperfunctie: is de klant veranderd?
# Bij SCD Type 2 vergelijken we eerst de inhoudelijke attributen.
# Alleen als er iets veranderd is, sluiten we de oude rij af
# en voegen we een nieuwe versie toe.

def klant_is_changed(source_row, dwh_row):
    return (
        str(source_row["naam"]) != str(dwh_row["naam"]) or
        str(source_row["adres"]) != str(dwh_row["adres"]) or
        str(source_row["woonplaats"]) != str(dwh_row["woonplaats"]) or
        str(source_row["geslacht"]) != str(dwh_row["geslacht"]) or
        str(source_row["geboortedatum"]) != str(dwh_row["geboortedatum"])
    )

# 5. Vooraf bepalen hoeveel records nieuw / gewijzigd / ongewijzigd zijn
# Dit is niet verplicht voor de ETL,
# maar wel handig voor logging en controle.

new_count = 0
changed_count = 0
preview_unchanged_count = 0

for _, src_row in df_klant_source.iterrows():
    klantnr = src_row["klantnr"]

    current_match = df_klant_dwh[df_klant_dwh["klantnr"] == klantnr]

    if current_match.empty:
        new_count += 1
    else:
        dwh_row = current_match.iloc[0]

        if klant_is_changed(src_row, dwh_row):
            changed_count += 1
        else:
            preview_unchanged_count += 1

logger.info(f"Aantal nieuwe klanten: {new_count}")
logger.info(f"Aantal gewijzigde klanten: {changed_count}")
logger.info(f"Aantal ongewijzigde klanten: {preview_unchanged_count}")

# 6. Echte ETL uitvoeren met SCD Type 2
# - nieuw -> INSERT
# - gewijzigd -> oude rij afsluiten + nieuwe rij INSERT
# - ongewijzigd -> niets doen

now_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

inserted_count = 0
updated_count = 0
unchanged_count = 0

logger.info("Start ETL voor Dim_Klant (SCD Type 2)")

for _, src_row in df_klant_source.iterrows():
    klantnr = src_row["klantnr"]

    current_match = df_klant_dwh[df_klant_dwh["klantnr"] == klantnr]


    # NIEUWE KLANT -> INSERT
    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Klant (
                klantnr,
                naam,
                adres,
                woonplaats,
                geslacht,
                geboortedatum,
                begintijd,
                eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, NULL)
        """, (
            int(src_row["klantnr"]),
            src_row["naam"],
            src_row["adres"],
            src_row["woonplaats"],
            src_row["geslacht"],
            src_row["geboortedatum"],
            now_ts
        ))

        inserted_count += 1

    # BESTAANDE KLANT -> check of gewijzigd
    else:
        dwh_row = current_match.iloc[0]

        if klant_is_changed(src_row, dwh_row):
            # Stap 1: oude versie afsluiten
            dwh_conn.execute("""
                UPDATE Dim_Klant
                SET eindtijd = ?
                WHERE klant_sk = ?
            """, (
                now_ts,
                int(dwh_row["klant_sk"])
            ))

            # Stap 2: nieuwe versie toevoegen
            dwh_conn.execute("""
                INSERT INTO Dim_Klant (
                    klantnr,
                    naam,
                    adres,
                    woonplaats,
                    geslacht,
                    geboortedatum,
                    begintijd,
                    eindtijd
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, NULL)
            """, (
                int(src_row["klantnr"]),
                src_row["naam"],
                src_row["adres"],
                src_row["woonplaats"],
                src_row["geslacht"],
                src_row["geboortedatum"],
                now_ts
            ))

            updated_count += 1

        else:
            # klant is niet veranderd
            unchanged_count += 1

# 7. Commit

dwh_conn.commit()

logger.info(
    f"Dim_Klant klaar | inserted={inserted_count}, "
    f"updated_scd2={updated_count}, unchanged={unchanged_count}, "
    f"conflicts={len(conflicts)}"
)

# 8. Controle
result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Klant
    ORDER BY klantnr, klant_sk
""", dwh_conn)

print(result_df)

2026-03-30 16:50:51.414 | INFO     | __main__:<module>:27 - Start ETL Dim_Klant (SCD Type 2)
2026-03-30 16:50:51.421 | INFO     | __main__:<module>:60 - Aantal klant-rijen uit accessoirebron: 20
2026-03-30 16:50:51.423 | INFO     | __main__:<module>:61 - Aantal klant-rijen uit fietsbron: 25


2026-03-30 16:50:51.482 | WARNING  | __main__:<module>:103 - Conflict klantnr=16
2026-03-30 16:50:51.487 | WARNING  | __main__:<module>:103 - Conflict klantnr=17
2026-03-30 16:50:51.491 | WARNING  | __main__:<module>:103 - Conflict klantnr=18
2026-03-30 16:50:51.493 | WARNING  | __main__:<module>:103 - Conflict klantnr=19
2026-03-30 16:50:51.497 | WARNING  | __main__:<module>:103 - Conflict klantnr=20
2026-03-30 16:50:51.503 | INFO     | __main__:<module>:108 - Aantal conflicten in brondata: 5
2026-03-30 16:50:51.505 | INFO     | __main__:<module>:109 - Aantal unieke klanten in source: 20
2026-03-30 16:50:51.529 | INFO     | __main__:<module>:134 - Aantal actuele klanten in DWH: 0
2026-03-30 16:50:51.540 | INFO     | __main__:<module>:177 - Aantal nieuwe klanten: 20
2026-03-30 16:50:51.543 | INFO     | __main__:<module>:178 - Aantal gewijzigde klanten: 0
2026-03-30 16:50:51.544 | INFO     | __main__:<module>:179 - Aantal ongewijzigde klanten: 0
2026-03-30 16:50:51.547 | INFO     | __ma

    klant_sk  klantnr            naam                 adres  woonplaats  \
0         21        1      Jan Jansen         Kerkstraat 12   Amsterdam   
1         22        2  Sophie de Boer           Lindelaan 8   Rotterdam   
2         23        3   Pieter Visser         Havenstraat 3    Den Haag   
3         24        4       Emma Smit          Boomgaard 22     Haarlem   
4         25        5      Tom Bakker        Stationsweg 44      Leiden   
5         26        6     Lisa Meijer         Dijkstraat 10     Zaandam   
6         27        7   Bart de Vries      Brouwersgracht 7       Delft   
7         28        8  Julia van Dijk         Plataanlaan 5       Hoorn   
8         29        9       Kevin Mol             Singel 99     Alkmaar   
9         30       10      Nina Groen        Waterstraat 16    Schiedam   
10        31       11    Daan Willems           Parklaan 31     Haarlem   
11        32       12         Eva Vos            Zomerweg 2  Zoetermeer   
12        33       13    

Dim_Accessoire (SCD TYPE 1) ROBBERT

In [35]:
# Wat willen we bereiken?
# We willen Dim_Accessoire in het DWH vullen met SCD Type 1 logica:
# - nieuw accessoire in SDM, nog niet in DWH -> INSERT
# - bestaand accessoire, maar gewijzigd -> UPDATE (overschrijven)
# - bestaand accessoire, niet gewijzigd -> niets doen

# business key = accessoirenr uit SDM
# surrogate key = accessoire_sk in DWH

# 1. Brondata ophalen per bron (horizontale samenvoeging)
# Voor Dim_Accessoire komt de data uit:
# - Accessoire_Verkoop_Accessoire + Accessoire_Verkoop_Leverancier
# - Accessoire_Inkoop_Accessoire + Accessoire_Inkoop_Leverancier

# We JOINEN eerst per bron, omdat accessoire- en leveranciersdata
# in aparte tabellen staan.

query_accessoire_verkoop = """
SELECT
    a.accessoirenr,
    a.naam,
    a.soort,
    l.leveranciernr,
    l.naam AS leveranciernaam,
    l.adres AS leverancieradres,
    l.woonplaats AS leverancierplaats
FROM Accessoire_Verkoop_Accessoire a
JOIN Accessoire_Verkoop_Leverancier l
    ON a.leverancier = l.leveranciernr
"""

query_accessoire_inkoop = """
SELECT
    a.accessoirenr,
    a.naam,
    a.soort,
    l.leveranciernr,
    l.naam AS leveranciernaam,
    l.adres AS leverancieradres,
    l.woonplaats AS leverancierplaats
FROM Accessoire_Inkoop_Accessoire a
JOIN Accessoire_Inkoop_Leverancier l
    ON a.leverancier = l.leveranciernr
"""

df_accessoire_verkoop = pd.read_sql_query(query_accessoire_verkoop, sdm_conn)
df_accessoire_inkoop = pd.read_sql_query(query_accessoire_inkoop, sdm_conn)

logger.info(f"Aantal accessoire-rijen uit verkoopbron: {len(df_accessoire_verkoop)}")
logger.info(f"Aantal accessoire-rijen uit inkoopbron: {len(df_accessoire_inkoop)}")

# 2. Verticale samenvoeging tot één bronset
# We zetten de rijen uit verkoop en inkoop onder elkaar.
# Daarna houden we per business key nog maar één record over.

# Combineer beide bronnen
df_all = pd.concat(
    [
        df_accessoire_verkoop.assign(bron="verkoop"),
        df_accessoire_inkoop.assign(bron="inkoop")
    ],
    ignore_index=True
)

# Groepeer per accessoirenr
df_accessoire_source = []
conflicts = []

for accessoirenr, group in df_all.groupby("accessoirenr"):
    if len(group) == 1:
        # maar 1 bron → gewoon nemen
        df_accessoire_source.append(group.iloc[0])
    else:
        # meerdere bronnen → check of ze gelijk zijn
        unieke = group.drop_duplicates(subset=[
            "naam", "soort",
            "leveranciernr", "leveranciernaam",
            "leverancieradres", "leverancierplaats"
        ])

        if len(unieke) == 1:
            df_accessoire_source.append(unieke.iloc[0])
        else:
            conflicts.append(accessoirenr)
            logger.warning(f"CONFLICT accessoirenr={accessoirenr}")
            print(group)

# maak dataframe
df_accessoire_source = pd.DataFrame(df_accessoire_source)
logger.info(f"Aantal unieke accessoires in source: {len(df_accessoire_source)}")

# 3. Actuele records uit het DWH ophalen
# We vergelijken met de huidige inhoud van Dim_Accessoire.
query_dwh_accessoire = """
SELECT
    accessoire_sk,
    accessoirenr,
    naam,
    soort,
    leveranciernr,
    leveranciernaam,
    leverancieradres,
    leverancierplaats,
    begintijd,
    eindtijd
FROM Dim_Accessoire
WHERE eindtijd IS NULL
"""

df_accessoire_dwh = pd.read_sql_query(query_dwh_accessoire, dwh_conn)

logger.info(f"Aantal actuele accessoires in DWH: {len(df_accessoire_dwh)}")

# 4. Helperfunctie: is het accessoire veranderd?
# Bij SCD Type 1 vergelijken we eerst of de inhoudelijke attributen veranderd zijn.
# Alleen dan voeren we een UPDATE uit.

def accessoire_is_changed(source_row, dwh_row):
    return (
        str(source_row["naam"]) != str(dwh_row["naam"]) or
        str(source_row["soort"]) != str(dwh_row["soort"]) or
        str(source_row["leveranciernr"]) != str(dwh_row["leveranciernr"]) or
        str(source_row["leveranciernaam"]) != str(dwh_row["leveranciernaam"]) or
        str(source_row["leverancieradres"]) != str(dwh_row["leverancieradres"]) or
        str(source_row["leverancierplaats"]) != str(dwh_row["leverancierplaats"])
    )

# 5. Vooraf bepalen hoeveel records nieuw / gewijzigd / ongewijzigd zijn
new_count = 0
changed_count = 0
preview_unchanged_count = 0

for _, src_row in df_accessoire_source.iterrows():
    accessoirenr = src_row["accessoirenr"]

    current_match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == accessoirenr]

    if current_match.empty:
        new_count += 1
    else:
        dwh_row = current_match.iloc[0]

        if accessoire_is_changed(src_row, dwh_row):
            changed_count += 1
        else:
            preview_unchanged_count += 1

logger.info(f"Aantal nieuwe accessoires: {new_count}")
logger.info(f"Aantal gewijzigde accessoires: {changed_count}")
logger.info(f"Aantal ongewijzigde accessoires: {preview_unchanged_count}")

# 6. Echte ETL uitvoeren met SCD Type 1
# - nieuw -> INSERT
# - gewijzigd -> UPDATE
# - ongewijzigd -> niets doen

now_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

inserted_count = 0
updated_count = 0
unchanged_count = 0

logger.info("Start ETL voor Dim_Accessoire (SCD Type 1)")

for _, src_row in df_accessoire_source.iterrows():
    accessoirenr = src_row["accessoirenr"]

    current_match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == accessoirenr]

    # Nieuw accessoire -> INSERT
    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Accessoire (
                accessoirenr,
                naam,
                soort,
                leveranciernr,
                leveranciernaam,
                leverancieradres,
                leverancierplaats,
                begintijd,
                eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (
            int(src_row["accessoirenr"]),
            src_row["naam"],
            src_row["soort"],
            int(src_row["leveranciernr"]),
            src_row["leveranciernaam"],
            src_row["leverancieradres"],
            src_row["leverancierplaats"],
            now_ts
        ))
        inserted_count += 1
    # Bestaand accessoire -> check of gewijzigd
    else:
        dwh_row = current_match.iloc[0]

        if accessoire_is_changed(src_row, dwh_row):
            # Bij SCD Type 1 overschrijven we de bestaande rij.
            dwh_conn.execute("""
                UPDATE Dim_Accessoire
                SET
                    naam = ?,
                    soort = ?,
                    leveranciernr = ?,
                    leveranciernaam = ?,
                    leverancieradres = ?,
                    leverancierplaats = ?
                WHERE accessoire_sk = ?
            """, (
                src_row["naam"],
                src_row["soort"],
                int(src_row["leveranciernr"]),
                src_row["leveranciernaam"],
                src_row["leverancieradres"],
                src_row["leverancierplaats"],
                int(dwh_row["accessoire_sk"])
            ))
            updated_count += 1
        else:
            unchanged_count += 1

dwh_conn.commit()

logger.info(
    f"Dim_Accessoire klaar | inserted={inserted_count}, "
    f"updated_scd1={updated_count}, unchanged={unchanged_count}"
)

# 7. Controle
result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Accessoire
    ORDER BY accessoirenr, accessoire_sk
""", dwh_conn)

print(result_df)

2026-03-30 14:15:07.161 | INFO     | __main__:<module>:49 - Aantal accessoire-rijen uit verkoopbron: 10
2026-03-30 14:15:07.162 | INFO     | __main__:<module>:50 - Aantal accessoire-rijen uit inkoopbron: 13
2026-03-30 14:15:07.175 | INFO     | __main__:<module>:90 - Aantal unieke accessoires in source: 13
2026-03-30 14:15:07.177 | INFO     | __main__:<module>:112 - Aantal actuele accessoires in DWH: 13
2026-03-30 14:15:07.184 | INFO     | __main__:<module>:148 - Aantal nieuwe accessoires: 0
2026-03-30 14:15:07.185 | INFO     | __main__:<module>:149 - Aantal gewijzigde accessoires: 0
2026-03-30 14:15:07.185 | INFO     | __main__:<module>:150 - Aantal ongewijzigde accessoires: 13
2026-03-30 14:15:07.186 | INFO     | __main__:<module>:163 - Start ETL voor Dim_Accessoire (SCD Type 1)
2026-03-30 14:15:07.194 | INFO     | __main__:<module>:227 - Dim_Accessoire klaar | inserted=0, updated_scd1=0, unchanged=13


    accessoire_sk  accessoirenr                      naam        soort  \
0               1             1              LED voorlamp  Verlichting   
1               2             2           LED achterlicht  Verlichting   
2               3             3  USB-oplaadbare fietslamp  Verlichting   
3               4             4              Ringslot AXA       Sloten   
4               5             5    Ketting met cijferslot       Sloten   
5               6             6         U-slot met beugel       Sloten   
6               7             7    Dubbele fietstas zwart       Tassen   
7               8             8   Enkele schouderfietstas       Tassen   
8               9             9      Waterdichte zadeltas       Tassen   
9              10            10      Kinderfietsmand roze       Tassen   
10             11            11  Reflecterende spaakclips  Verlichting   
11             12            12        Vouwbaar kabelslot       Sloten   
12             13            13       

Dim_Datum ROBBERT

In [36]:
# Wat willen we bereiken?
# We willen Dim_Datum in het DWH vullen.
# Voor Dim_Datum geldt:
# - nieuwe datum in SDM, nog niet in DWH -> INSERT
# - bestaande datum in DWH -> niets doen
# Voor Dim_Datum gebruiken we geen SCD Type 1 of Type 2,
# omdat datumwaarden zelf niet wijzigen.

# business key = datum
# surrogate key = datum_sk in DWH

# 1. Brondata ophalen uit de relevante feitbronnen=
# We halen de datums op uit:
# - Fiets_Verkoop
# - Accessoire_Verkoop
# - Onderhoud
# Voor Fct_Inkoop hebben we in het SDM geen echte datumkolom,
# maar maand + jaar. Daarom gebruiken we hier alleen de tabellen
# die een volledige datum bevatten.

query_fiets_verkoop_datum = """
SELECT datum
FROM Fiets_Verkoop
"""

query_accessoire_verkoop_datum = """
SELECT datum
FROM Accessoire_Verkoop
"""

query_onderhoud_datum = """
SELECT datum
FROM Onderhoud
"""

df_fiets_verkoop_datum = pd.read_sql_query(query_fiets_verkoop_datum, sdm_conn)
df_accessoire_verkoop_datum = pd.read_sql_query(query_accessoire_verkoop_datum, sdm_conn)
df_onderhoud_datum = pd.read_sql_query(query_onderhoud_datum, sdm_conn)

logger.info(f"Aantal datum-rijen uit Fiets_Verkoop: {len(df_fiets_verkoop_datum)}")
logger.info(f"Aantal datum-rijen uit Accessoire_Verkoop: {len(df_accessoire_verkoop_datum)}")
logger.info(f"Aantal datum-rijen uit Onderhoud: {len(df_onderhoud_datum)}")

# 2. Verticale samenvoeging tot één bronset
# We zetten alle datumrijen onder elkaar.
# Daarna houden we alleen unieke datums over.

df_datum_source = pd.concat(
    [df_fiets_verkoop_datum, df_accessoire_verkoop_datum, df_onderhoud_datum],
    ignore_index=True
)

df_datum_source = df_datum_source.drop_duplicates(subset=["datum"]).reset_index(drop=True)

logger.info(f"Aantal unieke datums in source: {len(df_datum_source)}")

# 3. Datumonderdelen afleiden
# We zetten de datumkolom om naar datetime,
# zodat we year, month en day kunnen afleiden.

df_datum_source["datum"] = pd.to_datetime(df_datum_source["datum"])
df_datum_source["year"] = df_datum_source["datum"].dt.year
df_datum_source["month"] = df_datum_source["datum"].dt.month
df_datum_source["day"] = df_datum_source["datum"].dt.day

# Voor opslag in SQLite zetten we datum weer om naar YYYY-MM-DD string
df_datum_source["datum"] = df_datum_source["datum"].dt.strftime("%Y-%m-%d")

# 4. Bestaande datums uit DWH ophalen
query_dwh_datum = """
SELECT
    datum_sk,
    datum,
    year,
    month,
    day
FROM Dim_Datum
"""

df_datum_dwh = pd.read_sql_query(query_dwh_datum, dwh_conn)

logger.info(f"Aantal datums in DWH: {len(df_datum_dwh)}")

# 5. Bepalen welke datums nieuw zijn
new_count = 0
existing_count = 0

for _, src_row in df_datum_source.iterrows():
    datum = src_row["datum"]

    current_match = df_datum_dwh[df_datum_dwh["datum"] == datum]

    if current_match.empty:
        new_count += 1
    else:
        existing_count += 1

logger.info(f"Aantal nieuwe datums: {new_count}")
logger.info(f"Aantal bestaande datums: {existing_count}")

# 6. Echte ETL uitvoeren
# Alleen nieuwe datums worden toegevoegd.

inserted_count = 0
logger.info("Start ETL voor Dim_Datum")

for _, src_row in df_datum_source.iterrows():
    datum = src_row["datum"]

    current_match = df_datum_dwh[df_datum_dwh["datum"] == datum]

    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Datum (
                datum,
                year,
                month,
                day
            )
            VALUES (?, ?, ?, ?)
        """, (
            src_row["datum"],
            int(src_row["year"]),
            int(src_row["month"]),
            int(src_row["day"])
        ))
        inserted_count += 1

dwh_conn.commit()
logger.info(f"Dim_Datum klaar | inserted={inserted_count}")

# 7. Controle

result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Datum
    ORDER BY datum
""", dwh_conn)

print(result_df)

2026-03-30 14:15:07.228 | INFO     | __main__:<module>:40 - Aantal datum-rijen uit Fiets_Verkoop: 150
2026-03-30 14:15:07.231 | INFO     | __main__:<module>:41 - Aantal datum-rijen uit Accessoire_Verkoop: 100
2026-03-30 14:15:07.233 | INFO     | __main__:<module>:42 - Aantal datum-rijen uit Onderhoud: 50
2026-03-30 14:15:07.240 | INFO     | __main__:<module>:55 - Aantal unieke datums in source: 196
2026-03-30 14:15:07.258 | INFO     | __main__:<module>:82 - Aantal datums in DWH: 201
2026-03-30 14:15:07.378 | INFO     | __main__:<module>:98 - Aantal nieuwe datums: 0
2026-03-30 14:15:07.379 | INFO     | __main__:<module>:99 - Aantal bestaande datums: 196
2026-03-30 14:15:07.380 | INFO     | __main__:<module>:105 - Start ETL voor Dim_Datum
2026-03-30 14:15:07.434 | INFO     | __main__:<module>:130 - Dim_Datum klaar | inserted=0


     datum_sk       datum  year  month  day
0         200  2024-01-01  2024      1    1
1           6  2024-01-02  2024      1    2
2         123  2024-01-06  2024      1    6
3          20  2024-01-08  2024      1    8
4         168  2024-01-10  2024      1   10
..        ...         ...   ...    ...  ...
196        61  2024-12-26  2024     12   26
197       183  2024-12-27  2024     12   27
198       172  2024-12-28  2024     12   28
199        95  2024-12-30  2024     12   30
200       119  2024-12-31  2024     12   31

[201 rows x 5 columns]


Fct_Verkoop ROBBERT


In [ ]:
# Wat willen we bereiken?
# We willen Fct_Verkoop vullen met verkoopgebeurtenissen uit:
# - Fiets_Verkoop
# - Accessoire_Verkoop

# In het DWH komt dit samen in één feittabel:
# Fct_Verkoop
# De feittabel bevat:
# - verkoopnr
# - verkoop_type
# - datum_sk
# - aantal
# - totaalprijs
# - standaardprijs
# - klant_sk
# - fiets_sk of accessoire_sk
# - monteur_sk

# 1. Brondata ophalen uit beide verkoopbronnen
# We voegen nu ook verkoop_type toe, zodat de samengestelde PK
# (verkoopnr, verkoop_type) correct gevuld kan worden.
logger.info("Ophalen Verkoop uit SDM")

# --- BRONDATA ---
query_fiets_verkoop = """
SELECT
    fv.datum,
    fv.aantal,
    fv.verkoopprijs AS totaalprijs,
    ff.standaardprijs,
    fv.klant,
    fv.fiets,
    fv.monteur,
    NULL AS accessoire
FROM Fiets_Verkoop fv
JOIN Fiets_Verkoop_Fiets ff
    ON fv.fiets = ff.fietsnr
"""

query_accessoire_verkoop = """
SELECT
    av.datum,
    av.aantal,
    av.verkoopprijs AS totaalprijs,
    aa.standaardprijs,
    av.klant,
    NULL AS fiets,
    av.monteur,
    av.accessoire
FROM Accessoire_Verkoop av
JOIN Accessoire_Verkoop_Accessoire aa
    ON av.accessoire = aa.accessoirenr
"""

df_verkoop_source = pd.concat([
    pd.read_sql_query(query_fiets_verkoop, sdm_conn),
    pd.read_sql_query(query_accessoire_verkoop, sdm_conn)
], ignore_index=True)

logger.info(f"Totaal bron: {len(df_verkoop_source)}")

# --- DIMENSIES ---
df_datum_dwh = pd.read_sql_query("SELECT datum_sk, datum FROM Dim_Datum", dwh_conn)

df_klant_dwh = pd.read_sql_query("""
SELECT klant_sk, klantnr FROM Dim_Klant WHERE eindtijd IS NULL
""", dwh_conn)

df_fiets_dwh = pd.read_sql_query("""
SELECT fiets_sk, fietsnr FROM Dim_Fiets WHERE eindtijd IS NULL
""", dwh_conn)

df_accessoire_dwh = pd.read_sql_query("""
SELECT accessoire_sk, accessoirenr FROM Dim_Accessoire WHERE eindtijd IS NULL
""", dwh_conn)

df_monteur_dwh = pd.read_sql_query("""
SELECT monteur_sk, monteurnr FROM Dim_Monteur WHERE eindtijd IS NULL
""", dwh_conn)

# --- BESTAANDE ---
df_verkoop_dwh = pd.read_sql_query("""
SELECT verkoop_type, datum_sk, aantal, klant_sk, fiets_sk, accessoire_sk
FROM Fct_Verkoop
""", dwh_conn)

bestaande_records = set(
    tuple(x) for x in df_verkoop_dwh.to_records(index=False)
)

# --- SAFE HELPERS ---
def safe_lookup(df, key_col, key_val, return_col, naam):
    match = df[df[key_col] == key_val]
    if match.empty:
        raise ValueError(f"{naam} niet gevonden: {key_val}")
    return int(match.iloc[0][return_col])

def get_datum_sk(d):
    return safe_lookup(df_datum_dwh, "datum", str(d), "datum_sk", "datum")

def get_klant_sk(k):
    return safe_lookup(df_klant_dwh, "klantnr", int(k), "klant_sk", "klant")

def get_fiets_sk(f):
    match = df_fiets_dwh[df_fiets_dwh["fietsnr"] == int(f)]
    return int(match.iloc[0]["fiets_sk"]) if not match.empty else None

def get_accessoire_sk(a):
    match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == int(a)]
    return int(match.iloc[0]["accessoire_sk"]) if not match.empty else None

def get_monteur_sk(m):
    return safe_lookup(df_monteur_dwh, "monteurnr", int(m), "monteur_sk", "monteur")

# --- ETL ---
logger.info("Start ETL Fct_Verkoop")

inserted = 0
skipped = 0
errors = 0

for _, row in df_verkoop_source.iterrows():
    try:
        datum_sk = get_datum_sk(row["datum"])
        klant_sk = get_klant_sk(row["klant"])
        monteur_sk = get_monteur_sk(row["monteur"])

        # type bepalen
        if pd.notna(row["fiets"]):
            verkoop_type = "fiets"
            fiets_sk = get_fiets_sk(row["fiets"])
            accessoire_sk = None
        else:
            verkoop_type = "accessoire"
            fiets_sk = None
            accessoire_sk = get_accessoire_sk(row["accessoire"])

        # validatie
        if (fiets_sk is None and accessoire_sk is None) or \
           (fiets_sk is not None and accessoire_sk is not None):
            logger.warning(f"Conflict: {row}")
            continue

        record_key = (
            verkoop_type,
            datum_sk,
            int(row["aantal"]),
            klant_sk,
            fiets_sk,
            accessoire_sk
        )

        if record_key in bestaande_records:
            skipped += 1
            continue

        dwh_conn.execute("""
            INSERT INTO Fct_Verkoop (
                verkoop_type,
                datum_sk,
                aantal,
                totaalprijs,
                standaardprijs,
                klant_sk,
                fiets_sk,
                accessoire_sk,
                monteur_sk
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            verkoop_type,
            datum_sk,
            int(row["aantal"]),
            float(row["totaalprijs"]),
            float(row["standaardprijs"]),
            klant_sk,
            fiets_sk,
            accessoire_sk,
            monteur_sk
        ))

        bestaande_records.add(record_key)  
        inserted += 1

    except Exception as e:
        logger.error(f"Fout bij row {row}: {e}")
        errors += 1

dwh_conn.commit()

logger.info(f"Fct_Verkoop klaar inserted={inserted}, skipped={skipped}, errors={errors}")

2026-03-30 14:15:07.452 | INFO     | __main__:<module>:22 - Ophalen Verkoop uit SDM
2026-03-30 14:15:07.456 | INFO     | __main__:<module>:60 - Totaal bron: 250
2026-03-30 14:15:07.463 | INFO     | __main__:<module>:116 - Start ETL Fct_Verkoop
2026-03-30 14:15:07.484 | ERROR    | __main__:<module>:186 - Fout bij row datum             2024-11-29
aantal                     1
totaalprijs            833.0
standaardprijs        490.58
klant                     17
fiets                     70
monteur                    7
accessoire              None
Name: 9, dtype: object: klant niet gevonden: 17
2026-03-30 14:15:07.490 | ERROR    | __main__:<module>:186 - Fout bij row datum             2024-08-26
aantal                     1
totaalprijs          2351.02
standaardprijs        890.64
klant                     20
fiets                      2
monteur                   10
accessoire              None
Name: 12, dtype: object: klant niet gevonden: 20
2026-03-30 14:15:07.492 | ERROR    | __main__:<

Dim_Fiets (SCD Type 2) MEES

In [38]:
logger.info("Ophalen Fiets uit SDM")

sdm_cursor.execute("""
SELECT DISTINCT
    fietsnr, soort, merk, type, kleur,
    fabrikantnr, naam, adres, plaats
FROM (
    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Fiets_Inkoop_Fiets f
    JOIN Fiets_Inkoop_Fabrikant fab ON f.fabrikant = fab.fabrikantnr

    UNION ALL

    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Fiets_Verkoop_Fiets f
    JOIN Fiets_Verkoop_Fabrikant fab ON f.fabrikant = fab.fabrikantnr

    UNION ALL

    SELECT 
        f.fietsnr, f.soort, f.merk, f.type, f.kleur,
        fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
    FROM Onderhoud_Fiets f
    JOIN Onderhoud_Fabrikant fab ON f.fabrikant = fab.fabrikantnr
)
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("""
SELECT 
    fietsnr, soort, merk, type, kleur,
    fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
    fiets_sk
FROM Dim_Fiets
WHERE eindtijd IS NULL
""")

dwh_data = {row[0]: row for row in dwh_cursor.fetchall()}
logger.info("Data opgehaald uit SDM en DWH.")

for row in source_data:
    (fietsnr, soort, merk, type_, kleur,
     fabrikantnr, fab_naam, fab_adres, fab_plaats) = row

    now = datetime.now()

    if fietsnr not in dwh_data:
        # nieuw
        logger.info(f"Nieuwe fiets: {fietsnr}")

        dwh_cursor.execute("""
        INSERT INTO Dim_Fiets (
            fietsnr, soort, merk, type, kleur,
            fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
            begintijd, eindtijd
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (fietsnr, soort, merk, type_, kleur,
              fabrikantnr, fab_naam, fab_adres, fab_plaats,
              now))

    else:
        # check wijzigingen
        dwh_row = dwh_data[fietsnr]

        (_, d_soort, d_merk, d_type, d_kleur,
         d_fabnr, d_fabnaam, d_fabadres, d_fabplaats,
         fiets_sk) = dwh_row

        if (soort, merk, type_, kleur,
            fabrikantnr, fab_naam, fab_adres, fab_plaats) != \
           (d_soort, d_merk, d_type, d_kleur,
            d_fabnr, d_fabnaam, d_fabadres, d_fabplaats):

            logger.info(f"Update in fiets: {fietsnr}")

            # 1. Sluit oude (voeg eindtijd toe)
            dwh_cursor.execute("""
            UPDATE Dim_Fiets
            SET eindtijd = ?
            WHERE fiets_sk = ?
            """, (now, fiets_sk))

            # 2. Maak nieuwe (met nieuwe gegevens en begintijd)
            dwh_cursor.execute("""
            INSERT INTO Dim_Fiets (
                fietsnr, soort, merk, type, kleur,
                fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
                begintijd, eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
            """, (fietsnr, soort, merk, type_, kleur,
                  fabrikantnr, fab_naam, fab_adres, fab_plaats,
                  now))

dwh_conn.commit()
logger.info("Fiets dimensie geupdate in DWH.")

2026-03-30 14:15:07.772 | INFO     | __main__:<module>:1 - Ophalen Fiets uit SDM
2026-03-30 14:15:07.774 | INFO     | __main__:<module>:44 - Data opgehaald uit SDM en DWH.
2026-03-30 14:15:07.775 | INFO     | __main__:<module>:80 - Update in fiets: 1
C:\Users\MeesR\AppData\Local\Temp\ipykernel_23612\2085707760.py:83: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  dwh_cursor.execute("""
C:\Users\MeesR\AppData\Local\Temp\ipykernel_23612\2085707760.py:90: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  dwh_cursor.execute("""
2026-03-30 14:15:07.778 | INFO     | __main__:<module>:80 - Update in fiets: 3
2026-03-30 14:15:07.778 | INFO     | __main__:<module>:80 - Update in fiets: 4
2026-03-30 14:15:07.779 | INFO     | __main__:<module>:80 - Update in fiets: 11
2026-03-30 14:15:07.780 | INFO   

Dim_Monteur (SCD Type 1) MEES

In [39]:
logger.info("Ophalen Monteur uit SDM")

sdm_cursor.execute("""
SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Accessoire_Verkoop_Monteur m
JOIN Accessoire_Verkoop_Filiaal f ON m.filiaal = f.filiaalnr

UNION ALL
                                   
SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Fiets_Verkoop_Monteur m
JOIN Fiets_Verkoop_Filiaal f ON m.filiaal = f.filiaalnr

UNION ALL

SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Onderhoud_Monteur m
JOIN Onderhoud_Filiaal f ON m.filiaal = f.filiaalnr
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("""
SELECT 
    monteurnr, naam, woonplaats, uurloon,
    filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
    monteur_sk
FROM Dim_Monteur
WHERE eindtijd IS NULL
""")

dwh_data = {row[0]: row for row in dwh_cursor.fetchall()}
logger.info("Data opgehaald uit SDM en DWH.")

processed_keys = set()

for row in source_data:
    (monteurnr, naam, woonplaats, uurloon,
     filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie) = row

    if monteurnr in processed_keys:
        continue
    processed_keys.add(monteurnr)

    now = datetime.now()

    if monteurnr not in dwh_data:
        # nieuwe regels worden toegevoegd
        logger.info(f"Nieuwe monteur: {monteurnr}")

        dwh_cursor.execute("""
        INSERT INTO Dim_Monteur (
            monteurnr, naam, woonplaats, uurloon,
            filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
            begintijd, eindtijd
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (monteurnr, naam, woonplaats, uurloon,
              filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
              now))

    else:
        # bestaande regels worden overschreven als er wijzigingen zijn
        (_, d_naam, d_woonplaats, d_uurloon,
         d_filiaalnr, d_filiaalnaam, d_filiaaladres, d_filiaalprovincie,
         monteur_sk) = dwh_data[monteurnr]

        if (naam, woonplaats, uurloon,
            filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie) != \
           (d_naam, d_woonplaats, d_uurloon,
            d_filiaalnr, d_filiaalnaam, d_filiaaladres, d_filiaalprovincie):

            logger.info(f"Update monteur (Type 1): {monteurnr}")

            dwh_cursor.execute("""
            UPDATE Dim_Monteur
            SET naam = ?, woonplaats = ?, uurloon = ?,
                filiaalnr = ?, filiaalnaam = ?, filiaaladres = ?, filiaalprovincie = ?
            WHERE monteur_sk = ?
            """, (naam, woonplaats, uurloon,
                  filiaalnr, filiaalnaam, filiaaladres, filiaalprovincie,
                  monteur_sk))
            
dwh_conn.commit()
logger.info("Monteur dimensie geupdate in DWH.")

2026-03-30 14:15:07.810 | INFO     | __main__:<module>:1 - Ophalen Monteur uit SDM
2026-03-30 14:15:07.811 | INFO     | __main__:<module>:57 - Data opgehaald uit SDM en DWH.
2026-03-30 14:15:07.812 | INFO     | __main__:<module>:109 - Monteur dimensie geupdate in DWH.


Fct_Onderhoud MEES

In [40]:
logger.info("Ophalen Onderhoud uit SDM")

sdm_cursor.execute("""
SELECT 
    o.onderhoudnr,
    o.datum,
    o.starttijd,
    o.eindtijd,
    o.fiets,
    o.monteur
FROM Onderhoud o
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("SELECT onderhoudnr FROM Fct_Onderhoud")
bestaande_ids = {row[0] for row in dwh_cursor.fetchall()}

def get_or_create_datum_sk(datum):
    dwh_cursor.execute("SELECT datum_sk FROM Dim_Datum WHERE datum = ?", (datum,))
    if r := dwh_cursor.fetchone():
        return r[0]

    from datetime import datetime
    d = datetime.strptime(datum, "%Y-%m-%d")

    dwh_cursor.execute("""
    INSERT INTO Dim_Datum (datum, year, month, day)
    VALUES (?, ?, ?, ?)
    """, (datum, d.year, d.month, d.day))

    return dwh_cursor.lastrowid


def get_fiets_sk(fietsnr):
    if fietsnr is None:
        return None
    dwh_cursor.execute("""
    SELECT fiets_sk FROM Dim_Fiets
    WHERE fietsnr = ? AND eindtijd IS NULL
    """, (fietsnr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


def get_monteur_sk(monteurnr):
    dwh_cursor.execute("""
    SELECT monteur_sk FROM Dim_Monteur
    WHERE monteurnr = ? AND eindtijd IS NULL
    """, (monteurnr,))
    result = dwh_cursor.fetchone()
    return result[0] if result else None


logger.info("Start ETL Fct_Onderhoud")

for row in source_data:
    onderhoudnr, datum, starttijd, eindtijd, fietsnr, monteurnr = row

    # skip bestaande
    if onderhoudnr in bestaande_ids:
        continue

    datum_sk = get_or_create_datum_sk(datum)
    fiets_sk = get_fiets_sk(fietsnr)
    monteur_sk = get_monteur_sk(monteurnr)

    # Check of alle keys bestaan
    if not all([datum_sk, fiets_sk, monteur_sk]):
        logger.warning(f"SK ontbreekt voor onderhoud {onderhoudnr}")
        continue

    # Insert fact
    dwh_cursor.execute("""
    INSERT INTO Fct_Onderhoud (
        onderhoudnr,
        datum_sk,
        starttijd,
        eindtijd,
        fiets_sk,
        monteur_sk
    )
    VALUES (?, ?, ?, ?, ?, ?)
    """, (onderhoudnr, datum_sk, starttijd, eindtijd, fiets_sk, monteur_sk))

dwh_conn.commit()
logger.success("Fct_Onderhoud load klaar")

2026-03-30 14:15:07.827 | INFO     | __main__:<module>:1 - Ophalen Onderhoud uit SDM
2026-03-30 14:15:07.829 | INFO     | __main__:<module>:55 - Start ETL Fct_Onderhoud
2026-03-30 14:15:07.830 | SUCCESS  | __main__:<module>:87 - Fct_Onderhoud load klaar


Fct_Inkoop MEES

In [41]:
logger.info("Ophalen Inkoop uit SDM") 

sdm_cursor.execute("""
SELECT 
    i.inkoopnr,
    DATE(i.inkoopjaar || '-' || printf('%02d', i.inkoopmaand) || '-01') AS datum,
    i.aantal,
    f.inkoopprijs,
    i.fiets,
    NULL as accessoire
FROM Fiets_Inkoop i
JOIN Fiets_Inkoop_Fiets f ON i.fiets = f.fietsnr

UNION ALL

SELECT 
    i.inkoopnr,
    DATE(i.inkoopjaar || '-' || printf('%02d', i.inkoopmaand) || '-01') AS datum,
    i.aantal,
    a.inkoopprijs,
    NULL as fiets,
    i.accessoire
FROM Accessoire_Inkoop i
JOIN Accessoire_Inkoop_Accessoire a ON i.accessoire = a.accessoirenr
""")

source_data = sdm_cursor.fetchall()

# LET OP: we gebruiken nu de SOURCE inkoopnr alleen voor incremental check
dwh_cursor.execute("SELECT inkoop_type, datum_sk, aantal FROM Fct_Inkoop")
bestaande_records = set(dwh_cursor.fetchall())


def get_fiets_sk(fietsnr):
    if fietsnr is None:
        return None
    dwh_cursor.execute("""
    SELECT fiets_sk FROM Dim_Fiets
    WHERE fietsnr = ? AND eindtijd IS NULL
    """, (fietsnr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


def get_accessoire_sk(accessoirenr):
    if accessoirenr is None:
        return None
    dwh_cursor.execute("""
    SELECT accessoire_sk FROM Dim_Accessoire
    WHERE accessoirenr = ? AND eindtijd IS NULL
    """, (accessoirenr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


logger.info("Start ETL Fct_Inkoop")

for row in source_data:
    source_inkoopnr, datum, aantal, inkoopprijs, fietsnr, accessoirenr = row

    datum_sk = get_or_create_datum_sk(datum)
    fiets_sk = get_fiets_sk(fietsnr)
    accessoire_sk = get_accessoire_sk(accessoirenr)

    # bepaal type
    if fiets_sk is not None:
        inkoop_type = "fiets"
    elif accessoire_sk is not None:
        inkoop_type = "accessoire"
    else:
        logger.warning(f"Ongeldige inkoop (geen match): {source_inkoopnr}")
        continue

    # validatie (extra check)
    if fiets_sk is not None and accessoire_sk is not None:
        logger.warning(f"Dubbele koppeling bij {source_inkoopnr}")
        continue

    totaalprijs = aantal * inkoopprijs

    # incremental check (zonder auto-id)
    record_key = (inkoop_type, datum_sk, aantal)
    if record_key in bestaande_records:
        continue

    dwh_cursor.execute("""
    INSERT INTO Fct_Inkoop (
        inkoop_type,
        datum_sk,
        aantal,
        totaalprijs,
        inkoopprijs,
        fiets_sk,
        accessoire_sk
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        inkoop_type,
        datum_sk,
        aantal,
        totaalprijs,
        inkoopprijs,
        fiets_sk,
        accessoire_sk
    ))

dwh_conn.commit()
logger.success("Fct_Inkoop load klaar")

2026-03-30 14:15:07.843 | INFO     | __main__:<module>:1 - Ophalen Inkoop uit SDM
2026-03-30 14:15:07.845 | INFO     | __main__:<module>:56 - Start ETL Fct_Inkoop
2026-03-30 14:15:07.858 | SUCCESS  | __main__:<module>:108 - Fct_Inkoop load klaar
